## How to read this notebook (What to look for)

This notebook focuses on **out-of-fold (OOF)** error patterns to understand *where* the model performs well and *where* it breaks down. When reviewing the plots below, pay attention to:

### 1) Residual vs. Prediction (OOF, raw scale)
- **Funnel shape** → heteroskedasticity (errors grow as price increases).
- **Mostly positive residuals at the high end** → the model is *under-predicting* expensive homes.
- **Mostly negative residuals at the low end** → the model is *over-predicting* cheaper homes.

### 2) Error by price segments (deciles)
- **MAE spikes in the top decile** → the model struggles on the luxury tail (large absolute dollar errors).
- **MAPE spikes in the bottom decile** → lower-priced homes are harder in *relative* terms (small dollar errors become large percentages).

### 3) Category slices (e.g., Neighborhood)
- If a few categories dominate MAE, it may indicate:
  - missing interactions (features behave differently across categories),
  - unmodeled local effects, or
  - data quality / feature sparsity issues concentrated in specific groups.


In [19]:
# ============================================
# Cell 0 — Robust repo root detection (showcase-safe)
# ============================================
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go

def find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    while True:
        # Prefer stable repo markers (not data files)
        if (cur / "src").exists() and (cur / "README.md").exists():
            return cur
        if cur == cur.parent:  # reached filesystem root
            break
        cur = cur.parent
    raise RuntimeError("Cannot locate repo root (expected to find 'src/' and 'README.md').")

REPO_ROOT = find_repo_root(Path.cwd())
print("REPO_ROOT =", REPO_ROOT)

# Ensure src import works even if notebook opened from subfolder
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

DATA_DIR = REPO_ROOT / "data" / "raw"
ARTIFACTS_DIR = REPO_ROOT / "artifacts"
REPORTS_DIR = ARTIFACTS_DIR / "reports"
REGISTRY_DIR = ARTIFACTS_DIR / "registry"
GLOBAL_ALIASES_PATH = REGISTRY_DIR / "_global" / "aliases.json"

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH  = DATA_DIR / "test.csv"

# Data is required
assert TRAIN_PATH.exists(), f"Missing {TRAIN_PATH}"
assert TEST_PATH.exists(), f"Missing {TEST_PATH}"

# Artifacts are outputs: create if missing (first run friendly)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
REGISTRY_DIR.mkdir(parents=True, exist_ok=True)
(REGISTRY_DIR / "_global").mkdir(parents=True, exist_ok=True)

# Global aliases may not exist before training: handle gracefully
if not GLOBAL_ALIASES_PATH.exists():
    # create an empty placeholder to avoid crashes
    GLOBAL_ALIASES_PATH.write_text(json.dumps({}, indent=2), encoding="utf-8")


train = pd.read_csv(TRAIN_PATH)
print("train:", train.shape)
train.head()


REPO_ROOT = C:\Users\23517\OneDrive\Documents\GitHub\house-prices-ml-pipeline
train: (1460, 81)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [20]:
from pathlib import Path
import pandas as pd
import numpy as np

# --------------------------------------------
# Cell 1 — Robust repo root detection (works in notebooks)
# --------------------------------------------
HERE = Path.cwd()

def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "src").exists() and (p / "data").exists():
            return p
    raise RuntimeError("Cannot find repo root (missing src/ and data/)")

REPO_ROOT = find_repo_root(HERE)

DATA_DIR    = REPO_ROOT / "data" / "raw"
REPORTS_DIR = REPO_ROOT / "artifacts" / "reports"

TRAIN_PATH = DATA_DIR / "train.csv"

print("REPO_ROOT =", REPO_ROOT)
print("TRAIN_PATH =", TRAIN_PATH)

assert TRAIN_PATH.exists(), f"Missing {TRAIN_PATH}"
assert REPORTS_DIR.exists(), f"Missing {REPORTS_DIR}"

train = pd.read_csv(TRAIN_PATH)
print("train shape:", train.shape)
train.head()


REPO_ROOT = c:\Users\23517\OneDrive\Documents\GitHub\house-prices-ml-pipeline
TRAIN_PATH = c:\Users\23517\OneDrive\Documents\GitHub\house-prices-ml-pipeline\data\raw\train.csv
train shape: (1460, 81)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [21]:
import plotly.io as pio
import plotly.graph_objects as go

# --- global theme ---
pio.templates["report"] = pio.templates["plotly_white"]

pio.templates["report"].layout.update(
    font=dict(family="Inter, Arial", size=14, color="#2c3e50"),
    title=dict(x=0.02, xanchor="left", font=dict(size=20)),
    margin=dict(l=70, r=40, t=80, b=70),
    hoverlabel=dict(bgcolor="white", font_size=13),
    xaxis=dict(showgrid=False, zeroline=False, ticks="outside"),
    yaxis=dict(gridcolor="rgba(0,0,0,0.06)", zeroline=False, ticks="outside"),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0.02),
)

pio.templates.default = "report"

def add_zero_line(fig, axis="y"):
    if axis == "y":
        fig.add_hline(y=0, line_width=2, line_color="rgba(44,62,80,0.6)")
    else:
        fig.add_vline(x=0, line_width=2, line_color="rgba(44,62,80,0.6)")
    return fig


In [22]:
# =========================
# Cell 2 — Discover OOF artifacts
# =========================
from pathlib import Path
import numpy as np

REPORTS_DIR = REPO_ROOT / "artifacts" / "reports"
assert REPORTS_DIR.exists(), f"Missing {REPORTS_DIR}"

def discover_oof_files(reports_dir: Path):
    patterns = [
        "*_oof.npy",
        "*_oof.csv",
        "*_oof.parquet",
    ]
    files = []
    for p in patterns:
        files.extend(reports_dir.glob(p))
    return sorted(files)

oof_files = discover_oof_files(REPORTS_DIR)

print("Found OOF files:")
for f in oof_files:
    print(" -", f.name)

assert len(oof_files) > 0, "No OOF files found in artifacts/reports/"


Found OOF files:
 - extratrees_oof.npy
 - lgbm_oof.npy
 - ridge_oof.npy
 - stacking_oof.npy
 - voting_mean_oof.npy
 - xgb_oof.npy


In [23]:
# ============================================
# Cell 2.5 — Load OOF files into oof_dfs dict
# ============================================
import numpy as np
import pandas as pd

def infer_model_name(path: Path) -> str:
    name = path.stem  # e.g. "xgb_oof"
    for suf in ["_oof_pred", "_oof_preds", "_oof"]:
        if name.endswith(suf):
            return name[: -len(suf)]
    return name.replace("_oof", "")

def load_oof_as_df(model_name: str, path: Path, train_df: pd.DataFrame) -> pd.DataFrame:
    # ground truth
    assert "SalePrice" in train_df.columns, "train.csv must contain SalePrice"
    if "Id" in train_df.columns:
        base = train_df[["Id", "SalePrice"]].copy()
    else:
        base = train_df[["SalePrice"]].copy()
        base["Id"] = np.arange(len(base))

    y_true_raw = base["SalePrice"].to_numpy(dtype=float)
    y_true_log = np.log1p(y_true_raw)

    # load pred
    if path.suffix.lower() == ".npy":
        y_pred = np.load(path).astype(float).ravel()
    elif path.suffix.lower() == ".csv":
        tmp = pd.read_csv(path)
        if tmp.shape[1] == 1:
            y_pred = tmp.iloc[:, 0].to_numpy(dtype=float)
        else:
            # try common column names
            pred_col = next((c for c in ["oof", "oof_pred", "pred", "prediction", "y_pred"] if c in tmp.columns), None)
            if pred_col is None:
                raise ValueError(f"Cannot infer pred column in {path.name}: {list(tmp.columns)}")
            y_pred = tmp[pred_col].to_numpy(dtype=float)
    else:
        raise ValueError(f"Unsupported OOF file type: {path.name}")

    if len(y_pred) != len(base):
        raise ValueError(f"Length mismatch for {model_name}: pred={len(y_pred)} vs train={len(base)}")

    # your project uses log1p(y) for training, so OOF is in log space
    y_pred_log = y_pred
    y_pred_raw = np.expm1(y_pred_log)

    df = pd.DataFrame({
        "Id": base["Id"].to_numpy(),
        "y_true_raw": y_true_raw,
        "y_pred_raw": y_pred_raw,
        "y_true_log": y_true_log,
        "y_pred_log": y_pred_log,
        "model": model_name,
    })
    df["residual_raw"] = df["y_true_raw"] - df["y_pred_raw"]
    df["residual_log"] = df["y_true_log"] - df["y_pred_log"]
    df["abs_error_raw"] = np.abs(df["residual_raw"])
    df["abs_error_log"] = np.abs(df["residual_log"])
    df["ape_raw"] = df["abs_error_raw"] / np.maximum(df["y_true_raw"], 1.0)
    return df

# Build mapping and load
model_to_file = {infer_model_name(f): f for f in oof_files}
print("Models detected:", sorted(model_to_file.keys()))

oof_dfs = {}
for m, f in model_to_file.items():
    try:
        oof_dfs[m] = load_oof_as_df(m, f, train)
        print(f"[OK] {m}: {f.name} -> {oof_dfs[m].shape}")
    except Exception as e:
        print(f"[SKIP] {m}: {f.name} -> {e}")

assert len(oof_dfs) > 0, "No usable OOF dfs loaded. Check SKIP errors above."


Models detected: ['extratrees', 'lgbm', 'ridge', 'stacking', 'voting_mean', 'xgb']
[OK] extratrees: extratrees_oof.npy -> (1460, 11)
[OK] lgbm: lgbm_oof.npy -> (1460, 11)
[OK] ridge: ridge_oof.npy -> (1460, 11)
[OK] stacking: stacking_oof.npy -> (1460, 11)
[OK] voting_mean: voting_mean_oof.npy -> (1460, 11)
[OK] xgb: xgb_oof.npy -> (1460, 11)


In [24]:
# ============================================
# Cell 3 — Pick a model (simple selector variable)
# ============================================
# Choose one from: sorted(oof_dfs.keys())
MODEL = "voting_mean"   # global best (cv_rmse=0.12705)
df = oof_dfs[MODEL].copy()

df.describe().T.head(12)


,count,mean,std,min,25%,50%,75%,max
Id,1460.0,730.500000,421.610009,1.000000,365.750000,730.500000,1095.250000,1460.000000
y_true_raw,1460.0,180921.195890,79442.502883,34900.000000,129975.000000,163000.000000,214000.000000,755000.000000
y_pred_raw,1460.0,179337.635423,73849.205543,54359.195953,129983.986846,162106.295647,210446.205443,771716.217869
y_true_log,1460.0,12.024057,0.399449,10.460271,11.775105,12.001512,12.273736,13.534474
y_pred_log,1460.0,12.025214,0.371283,10.903387,11.775174,11.996014,12.256990,13.556373
residual_raw,1460.0,1583.560467,31190.695823,-611716.217869,-7614.560491,862.081095,9666.318982,310248.757302
residual_log,1460.0,-0.001157,0.127090,-1.573438,-0.046231,0.004871,0.056152,0.538609
abs_error_raw,1460.0,14444.741016,27687.096925,16.015705,3880.870839,8507.243547,16866.382451,611716.217869
abs_error_log,1460.0,0.078150,0.100208,0.000080,0.023772,0.051869,0.096847,1.573438
ape_raw,1460.0,0.082672,0.158190,0.000080,0.023846,0.052000,0.096730,3.823226


OOF Error Summary

The OOF predictions show strong overall calibration in log space. The mean predicted log SalePrice (12.025) closely matches the mean true log SalePrice (12.024), and both the mean and median residuals are near zero, indicating no systematic over- or under-prediction.

In terms of relative accuracy, the median absolute percentage error is approximately 5%, and 75% of observations have percentage errors below 10%, suggesting that the model performs well for the majority of homes.

However, the maximum absolute and percentage errors are driven by a small number of extreme cases, likely corresponding to very high-priced properties. This suggests that while the model captures general market trends effectively, prediction uncertainty increases in the upper price range due to limited data and higher heterogeneity.

In [25]:
# ============================================
# Cell 4 — Residual vs Prediction (raw scale)
# ============================================
# Add a visual focus: highlight top-priced predictions
thr = np.quantile(df["y_pred_raw"], 0.95)  # top 5% predicted price
df["_tier"] = np.where(df["y_pred_raw"] >= thr, "Top 5% predicted price", "Others")

fig = px.scatter(
    df,
    x="y_pred_raw",
    y="residual_raw",
    color="_tier",
    hover_data=["Id", "y_true_raw", "y_pred_raw", "ape_raw"],
    title=f"Residual vs Prediction (OOF, raw) — {MODEL}",
)

fig.update_traces(marker=dict(size=6, opacity=0.45))
fig.update_layout(
    xaxis_title="Predicted SalePrice (raw scale)",
    yaxis_title="Residual = True - Predicted (raw scale)",
)
add_zero_line(fig, "y")
fig.show()



Residuals vs Predicted Price (OOF, raw scale)

The residual–prediction plot shows that errors are generally centered around zero, confirming the absence of systematic bias in the model. For the majority of properties, especially those in the low- to mid-price range, residuals are tightly clustered, indicating stable and accurate predictions.

However, as predicted prices increase, the variance of residuals also increases, revealing heteroskedasticity. This pattern suggests that prediction uncertainty grows for high-priced homes. A small number of extreme negative residuals appear in the upper price range, which explains the large maximum absolute errors observed in the summary statistics. These cases likely reflect the greater heterogeneity and limited sample size in the high-price segment of the market.

In [26]:
# ============================================
# Cell 5 — Residual distributions (raw + log)
# ============================================

# --- raw residual: long tail, show trimmed view for readability ---
lo, hi = np.quantile(df["residual_raw"], [0.01, 0.99])

fig1 = px.histogram(
    df,
    x="residual_raw",
    nbins=70,
    title=f"Residual Distribution (raw, 1–99% trimmed) — {MODEL}",
)
fig1.update_xaxes(range=[lo, hi], title="Residual (raw)")
fig1.update_layout(yaxis_title="Count", bargap=0.05)
add_zero_line(fig1, "x")
fig1.show()

# --- log residual: aligns with competition metric ---
fig2 = px.histogram(
    df,
    x="residual_log",
    nbins=70,
    title=f"Residual Distribution (log1p target space) — {MODEL}",
)
fig2.update_xaxes(title="Residual (log1p)")
fig2.update_layout(yaxis_title="Count", bargap=0.05)
add_zero_line(fig2, "x")
fig2.show()



Overall, the voting_mean ensemble demonstrates strong and well-calibrated performance on out-of-fold predictions.
Errors are small and symmetrically distributed in log space, aligning well with the competition’s evaluation metric.
The main weakness lies in a small number of extremely high-priced properties, where limited sample size and feature sparsity lead to underestimation.
Given the negligible impact of these outliers on log-RMSE, the current model is considered robust and suitable for final submission.

In [27]:
def nice_bin_label(interval) -> str:
    # interval is pandas.Interval
    left = int(interval.left)
    right = int(interval.right)
    return f"${left/1000:.0f}k–{right/1000:.0f}k"


In [28]:
# ============================================
# Cell 6 — Error by Price Segment (deciles)
# ============================================
q = pd.qcut(df["y_true_raw"], q=10, duplicates="drop")
seg = df.groupby(q, observed=True).agg(
    n=("Id", "count"),
    mae=("abs_error_raw", "mean"),
    mape=("ape_raw", "mean"),
    median_true=("y_true_raw", "median"),
).reset_index()

seg["segment_label"] = seg[q.name].apply(nice_bin_label)

# --- MAE ---
worst_mae_idx = seg["mae"].idxmax()
seg["_mae_flag"] = "Typical"
seg.loc[worst_mae_idx, "_mae_flag"] = "Worst segment (highest MAE)"

fig = px.bar(
    seg,
    x="segment_label",
    y="mae",
    color="_mae_flag",
    hover_data=["n", "mape", "median_true"],
    title=f"MAE by Price Segment (OOF) — {MODEL}",
)
fig.update_layout(
    xaxis_title="True SalePrice deciles",
    yaxis_title="MAE (raw dollars)",
)
fig.update_xaxes(tickangle=-30)
fig.show()

# --- MAPE ---
worst_mape_idx = seg["mape"].idxmax()
seg["_mape_flag"] = "Typical"
seg.loc[worst_mape_idx, "_mape_flag"] = "Worst segment (highest MAPE)"

fig = px.bar(
    seg,
    x="segment_label",
    y="mape",
    color="_mape_flag",
    hover_data=["n", "mae", "median_true"],
    title=f"MAPE by Price Segment (OOF) — {MODEL}",
)
fig.update_layout(
    xaxis_title="True SalePrice deciles",
    yaxis_title="MAPE",
)
fig.update_xaxes(tickangle=-30)
fig.show()


Error Analysis by Price Segment (OOF)

The segment-level error analysis reveals a clear difference between absolute and relative model performance across price ranges.

From the MAE perspective, prediction errors increase steadily as house prices rise. Low- and mid-priced homes exhibit relatively small absolute errors (approximately $8k–$15k), while the highest price segment shows substantially larger errors, exceeding $30k. This behavior is expected, as higher-priced properties tend to be more heterogeneous and harder to model precisely in absolute dollar terms.

In contrast, the MAPE results highlight an opposite pattern. The lowest price segment exhibits the highest relative error, with MAPE close to 18%, indicating that even modest absolute deviations represent a large proportion of the true sale price. Middle-priced homes achieve the most stable relative performance, with MAPE generally below 7%. Although absolute errors increase for high-priced homes, their relative errors remain controlled and do not grow proportionally.

Overall, these results suggest that the model performs most consistently in relative terms for mid-range properties, while low-priced homes suffer from higher proportional error and high-priced homes incur larger absolute deviations. This trade-off between absolute and relative accuracy is typical in housing price models and reflects inherent variability across different market segments.

In [29]:
# ============================================
# Cell 7 — Top error cases (inspect)
# ============================================
top = df.sort_values("abs_error_raw", ascending=False).head(25)
top[["Id", "y_true_raw", "y_pred_raw", "abs_error_raw", "ape_raw"]].reset_index(drop=True)


,Id,y_true_raw,y_pred_raw,abs_error_raw,ape_raw
0,1299,160000.0,771716.217869,611716.217869,3.823226
1,524,184750.0,680095.724951,495345.724951,2.681168
2,1183,745000.0,434751.242698,310248.757302,0.416441
3,692,755000.0,542942.722078,212057.277922,0.280871
4,804,582933.0,422768.854040,160164.145960,0.274756
5,689,392000.0,235544.119771,156455.880229,0.399122
6,899,611657.0,456450.927288,155206.072712,0.253747
7,1325,147000.0,294171.069253,147171.069253,1.001164
8,314,375000.0,243318.705763,131681.294237,0.351150
9,186,475000.0,345242.104348,129757.895652,0.273175


In [30]:
# ============================================
# Cell 8 — Join features back for slice-and-dice
# ============================================
features = train.drop(columns=["SalePrice"], errors="ignore").copy()
if "Id" not in features.columns:
    features["Id"] = np.arange(len(features))

df_full = df.merge(features, on="Id", how="left")
print("df_full:", df_full.shape)

df_full.head()


df_full: (1460, 91)


,Id,y_true_raw,y_pred_raw,y_true_log,y_pred_log,model,residual_raw,residual_log,abs_error_raw,abs_error_log,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,1,208500.0,209683.449446,12.247699,12.253359,voting_mean,-1183.449446,-0.005660,1183.449446,0.005660,...,0,0,NaN,NaN,NaN,0,2,2008,WD,Normal
1,2,181500.0,183536.285280,12.109016,12.120173,voting_mean,-2036.285280,-0.011157,2036.285280,0.011157,...,0,0,NaN,NaN,NaN,0,5,2007,WD,Normal
2,3,223500.0,214806.726912,12.317171,12.277499,voting_mean,8693.273088,0.039673,8693.273088,0.039673,...,0,0,NaN,NaN,NaN,0,9,2008,WD,Normal
3,4,140000.0,178873.413402,11.849405,12.094439,voting_mean,-38873.413402,-0.245034,38873.413402,0.245034,...,0,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml
4,5,250000.0,303761.510755,12.429220,12.624001,voting_mean,-53761.510755,-0.194781,53761.510755,0.194781,...,0,0,NaN,NaN,NaN,0,12,2008,WD,Normal


In [31]:
# ============================================
# Cell 9 — Segment error by Neighborhood (better style)
# ============================================
cat_col = "Neighborhood" if "Neighborhood" in df_full.columns else None

if cat_col is not None:
    g = df_full.groupby(cat_col, observed=True).agg(
        n=("Id", "count"),
        mae=("abs_error_raw", "mean"),
        mape=("ape_raw", "mean"),
        median_price=("y_true_raw", "median"),
    ).reset_index().sort_values("mae", ascending=False)

    g_top = g.head(25).copy()
    g_top = g_top.sort_values("mae", ascending=True)  # so the biggest is on top in horizontal bar

    fig = px.bar(
        g_top,
        x="mae",
        y=cat_col,
        orientation="h",
        color="mae",
        color_continuous_scale="Blues",
        hover_data=["n", "mape", "median_price"],
        title=f"Top 25 Neighborhoods by MAE (OOF) — {MODEL}",
    )
    fig.update_layout(
        xaxis_title="MAE (raw dollars)",
        yaxis_title="Neighborhood",
        coloraxis_showscale=False,
    )
    fig.show()
else:
    print("Neighborhood not found.")



Error Analysis by Neighborhood (OOF)

The neighborhood-level MAE analysis reveals substantial variation in model performance across different geographic areas. A small number of neighborhoods, such as NoRidge and StoneBr, exhibit notably higher absolute errors, exceeding $30k, while many other neighborhoods show much lower MAE, often below $10k.

These high-error neighborhoods tend to correspond to areas with higher-priced or more heterogeneous housing stock, where properties differ significantly in size, quality, and amenities. Such structural diversity makes it more difficult for the model to capture consistent pricing patterns using available features. In contrast, neighborhoods with lower MAE likely contain more homogeneous housing characteristics, allowing the model to generalize more effectively.

Overall, this result suggests that model error is not uniformly distributed spatially. Certain neighborhoods systematically contribute a disproportionate share of total prediction error. This insight highlights potential directions for further improvement, such as neighborhood-specific feature engineering, interaction terms, or localized modeling strategies for high-variance areas.

In [32]:
# ============================================
# Cell 10 — Compare all models (OOF metrics)
# ============================================
def rmse_vec(x):
    x = np.asarray(x, dtype=float)
    return float(np.sqrt(np.mean(x**2)))

summary = []
for m, d in oof_dfs.items():
    summary.append({
        "model": m,
        "RMSE_raw": rmse_vec(d["residual_raw"]),
        "MAE_raw": float(d["abs_error_raw"].mean()),
        "MAPE_raw": float(d["ape_raw"].mean()),
        "RMSE_log": rmse_vec(d["residual_log"]),
        "MAE_log": float(d["abs_error_log"].mean()),
    })

summary_df = pd.DataFrame(summary).sort_values("RMSE_log").reset_index(drop=True)
summary_df


,model,RMSE_raw,MAE_raw,MAPE_raw,RMSE_log,MAE_log
0,voting_mean,31220.199031,14444.741016,0.082672,0.127051,0.078150
1,xgb,28977.165905,14929.278756,0.085271,0.127126,0.081755
2,stacking,37403.003420,14535.283084,0.083591,0.129827,0.077733
3,lgbm,30438.563578,16203.395952,0.092512,0.135996,0.088416
4,ridge,58043.393797,15980.109929,0.091957,0.140888,0.082251
5,extratrees,31048.851170,17032.313088,0.097444,0.142796,0.093267


In [33]:
# ============================================
# Cell 11 — Model Comparison (log-space) + highlight selected
# ============================================
chosen = MODEL  # "voting_mean"

fig = px.scatter(
    summary_df,
    x="RMSE_log",
    y="MAE_log",
    text="model",
    hover_data=["RMSE_raw", "MAE_raw", "MAPE_raw"],
    title="Model Comparison (OOF): log-space metrics",
)

fig.update_traces(marker=dict(size=14, opacity=0.75))
fig.update_layout(xaxis_title="RMSE (log1p space)", yaxis_title="MAE (log1p space)")

# highlight chosen
row = summary_df[summary_df["model"] == chosen].iloc[0]
fig.add_trace(
    go.Scatter(
        x=[row["RMSE_log"]],
        y=[row["MAE_log"]],
        mode="markers",
        marker=dict(size=20, symbol="star"),
        name="Selected model",
        showlegend=True,
    )
)
fig.add_annotation(
    x=row["RMSE_log"],
    y=row["MAE_log"],
    text=f"Selected: {chosen}",
    showarrow=True,
    arrowhead=2,
    ax=40,
    ay=-40,
)

fig.show()


Model Comparison in Log Space (OOF)

This figure compares all candidate models using out-of-fold error metrics in log-transformed target space, with RMSE_log on the x-axis and MAE_log on the y-axis. Lower values on both axes indicate better overall predictive performance and more stable relative errors across the price range.

Among all evaluated models, voting_mean achieves the best overall trade-off, occupying the lower-left region of the plot. While the XGBoost model shows slightly competitive RMSE_log, it exhibits a higher MAE_log, indicating less stable absolute deviations in relative terms. The stacking model performs comparably but does not improve upon voting_mean on either metric. Other models, such as LightGBM, Ridge, and ExtraTrees, are clearly dominated, showing higher error levels on both axes.

These results support the selection of voting_mean as the final model, as it delivers the most balanced and robust performance across error metrics. In particular, its lower MAE_log suggests better consistency across different price scales, which is especially important for housing price prediction where relative error matters more than absolute error.